# Bölüm 18 — IBM SPSS Uygulamaları — Kontrol Defteri

Bu defter SPSS'in yerine geçmez. Amaç, `analiz.sps` ile SPSS'te elde ettiğiniz temel sayıları bağımsız bir Python kontrolüyle karşılaştırmaktır. Önce `VERI.md`, `SPSS-KONTROL-LISTESI.md` ve `GOREVLER.md` dosyalarını okuyun; en sonda `COZUMLER.md` ve `RAPORLAMA.md` ile kontrol edin.

In [ ]:
from pathlib import Path
import pandas as pd
from scipy import stats
HERE = Path.cwd()
print('Klasör:', HERE)

## 1. Veri bütünlüğü
SPSS içe aktarımı sonrasında gördüğünüz satır sayılarını burada karşılaştırın.

In [ ]:
sleep = pd.read_csv(HERE/'sleep.csv')
tooth = pd.read_csv(HERE/'ToothGrowth.csv')
sleep_eksik = pd.read_csv(HERE/'sleep_eksik.csv')
print('sleep:', sleep.shape, 'ID:', sleep.ID.nunique())
print('ToothGrowth:', tooth.shape)
print('sleep_eksik:', sleep_eksik.shape, 'eksik extra:', sleep_eksik.extra.isna().sum())

## 2. Eşli test kontrolü
SPSS'te ID üzerinden geniş biçime dönüştürdükten sonra `group2-group1` farkını kullanın.

In [ ]:
wide = sleep.pivot(index='ID', columns='group', values='extra')
diff = wide[2] - wide[1]
print('Geçerli çift:', diff.notna().sum())
print('Ortalama fark:', diff.mean())
print(stats.ttest_rel(wide[2], wide[1]))

## 3. `dose=1` filtre kontrolü
SPSS'te filtre sonrası aktif N'nin 20 olduğunu doğrulayın.

In [ ]:
dose1 = tooth[tooth.dose.eq(1.0)]
oj = dose1.loc[dose1.supp.eq('OJ'), 'len']
vc = dose1.loc[dose1.supp.eq('VC'), 'len']
print('Filtre sonrası N:', len(dose1), 'OJ:', len(oj), 'VC:', len(vc))
print('Ortalamalar:', oj.mean(), vc.mean())
print('Welch:', stats.ttest_ind(oj, vc, equal_var=False))

## 4. Eksik eş deneyi
Dosyada 20 satır kalmasına rağmen geçerli çift sayısının değiştiğini gösterin.

In [ ]:
wide_m = sleep_eksik.pivot(index='ID', columns='group', values='extra').dropna()
print('Geçerli çift:', len(wide_m))
print('Ortalama fark:', (wide_m[2]-wide_m[1]).mean())
print(stats.ttest_rel(wide_m[2], wide_m[1]))

## 5. Yorum alanı
Kendi SPSS çıktınız için şu soruları yanıtlayın:

- Aktif filtre/ağırlık/Split File durumu neydi?
- Analiz birimi neydi?
- Hangi SPSS tablosundan hangi sayı alındı?
- Raporlanan n neden dosya satır sayısından farklı olabilir?
- Sonuç hangi sınırların ötesine genellenemez?

## 6. Son kontrol
`GOREVLER.md` → kendi SPSS çözümünüz → `analiz.sps` → `COZUMLER.md` → `beklenen.json` → `RAPORLAMA.md`.

Python kontrolü ile SPSS sonucu farklıysa önce filtre, ağırlık, Split File, eksik değer, grup yönü ve veri dönüşümünü denetleyin.